# EnterpriseSim: GRPO Training for Customer Support Agents

**Goal**: Train a language model to be a better customer support agent using Group Relative Policy Optimization (GRPO).

**Pipeline**:
1. **Connect** to the live EnterpriseSim environment (OpenEnv on HuggingFace Spaces)
2. **Collect** training data by running episodes with a behavior policy
3. **Define** reward functions that capture what "good support" looks like
4. **Train** with GRPO using TRL + LoRA
5. **Evaluate** the trained model against the base model

**Environment**: [EnterpriseSim Customer Support](https://huggingface.co/spaces/jjmachan/enterprise-sim-support) — an OpenEnv RL environment where an agent handles customer support tickets using tools (lookup_customer, check_order, send_reply, update_ticket). A simulated customer (LLM-powered) responds with satisfaction signals.

## 1. Setup & Install

In [ ]:
!pip install -q trl peft transformers datasets openai torch accelerate
!pip install -q "openenv-core[core]>=0.2.1"

In [ ]:
import os
from google.colab import userdata

# Set your OpenAI API key (used by the environment's simulated customer agent)
# Add it in Colab: Settings > Secrets > OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 2. Connect to the Environment

The EnterpriseSim environment runs on HuggingFace Spaces. We connect via WebSocket using the OpenEnv client protocol. The environment exposes 4 tools:

- **lookup_customer** — Get customer profile, order history, VIP status
- **check_order** — Get order details, shipping status, tracking
- **send_reply** — Reply to the customer (triggers simulated customer response)
- **update_ticket** — Update ticket status (open/resolved/escalated)

In [ ]:
from typing import Any, Dict
from openenv.core.env_server.mcp_types import CallToolAction
from openenv.core.mcp_client import MCPToolClient

ENV_URL = "https://jjmachan-enterprise-sim-support.hf.space"


class SupportStepResult:
    """Rich step result that exposes all SupportObservation fields."""

    def __init__(self, raw: Dict[str, Any]):
        self._raw = raw
        obs = raw.get("observation", {})
        self.customer_message: str = obs.get("customer_message", "")
        self.tool_result: str = obs.get("tool_result", "")
        self.tool_name: str = obs.get("tool_name", "")
        self.ticket_context: str = obs.get("ticket_context", "")
        self.ticket_id: int = obs.get("ticket_id", 0)
        self.customer_id: str = obs.get("customer_id", "")
        self.satisfaction: float = obs.get("satisfaction", 0.0)
        self.satisfaction_delta: float = obs.get("satisfaction_delta", 0.0)
        self.resolved: bool = obs.get("resolved", False)
        self.step_count: int = obs.get("step_count", 0)
        self.episode_id: str = obs.get("episode_id", "")
        self.reward: float = raw.get("reward", 0.0)
        self.done: bool = raw.get("done", False)


class CustomerSupportEnv(MCPToolClient):
    """Client for the EnterpriseSim customer support environment."""

    def reset(self, **kwargs: Any) -> SupportStepResult:
        message = {"type": "reset", "data": kwargs}
        response = self._send_and_receive(message)
        return SupportStepResult(response.get("data", {}))

    def call_tool(self, name: str, **kwargs: Any) -> SupportStepResult:
        action = CallToolAction(tool_name=name, arguments=kwargs)
        payload = self._step_payload(action)
        message = {"type": "step", "data": payload}
        response = self._send_and_receive(message)
        return SupportStepResult(response.get("data", {}))

In [ ]:
# Quick test: connect, reset, and take one action
env = CustomerSupportEnv(base_url=ENV_URL)
with env:
    obs = env.reset()
    print(f"Customer: {obs.customer_message[:200]}")
    print(f"Ticket ID: {obs.ticket_id}, Customer ID: {obs.customer_id}")
    print(f"Satisfaction: {obs.satisfaction}")
    print()

    # Look up the customer
    obs = env.call_tool("lookup_customer", customer_id=obs.customer_id)
    print(f"Tool result: {obs.tool_result[:300]}")

print("\nEnvironment is working!")

## 3. Collect Training Data

We run episodes against the environment using a behavior policy (OpenAI API) to collect trajectories. Each trajectory step becomes a GRPO training example: the conversation so far (prompt) + metadata for reward computation (answer).

The model must output tool calls in this XML format:
```xml
<tool_call>
<function=tool_name>
<parameter=param_name>value</parameter>
</function>
</tool_call>
```

In [ ]:
import re
import json
import time
from openai import OpenAI
from datasets import Dataset

# --- Tool call parsing (Qwen XML format) ---

TOOL_CALL_RE = re.compile(
    r"<tool_call>\s*<function=(\w+)>(.*?)</function>\s*</tool_call>", re.DOTALL
)
PARAM_RE = re.compile(r"<parameter=(\w+)>(.*?)</parameter>", re.DOTALL)
VALID_TOOLS = {"lookup_customer", "check_order", "send_reply", "update_ticket"}


def parse_tool_call(text: str):
    """Extract tool name and arguments from XML tool call format."""
    match = TOOL_CALL_RE.search(text)
    if not match:
        return None, None
    tool_name = match.group(1).strip()
    args = {}
    for pm in PARAM_RE.finditer(match.group(2)):
        key = pm.group(1).strip()
        val = pm.group(2).strip()
        if key == "ticket_id":
            try:
                val = int(val)
            except ValueError:
                pass
        args[key] = val
    return tool_name, args


# --- System prompt ---

SYSTEM_PROMPT = """You are a Customer Support Representative at Office Furniture Co. Help customers by investigating their issues and providing concrete solutions.

## Available Tools

### lookup_customer
Look up a customer profile with order and ticket history summary.
Parameters:
  - customer_id (string, optional): Customer ID
  - customer_name (string, optional): Customer name

### check_order
Get full order details including items, status, and shipping info.
Parameters:
  - order_id (string, required): Order ID

### send_reply
Send a reply to a customer on a ticket. This will trigger a customer response.
Parameters:
  - ticket_id (integer, required): Ticket ID
  - message (string, required): Your reply message

### update_ticket
Update ticket status and/or add internal notes.
Parameters:
  - ticket_id (integer, required): Ticket ID
  - status (string, optional): New status (open/resolved/escalated)
  - notes (string, optional): Internal notes

## Company Policies (Summary)
- 30-day return policy, 15% restocking fee for opened items
- Damaged items: free replacement or full refund within 48 hours
- Standard shipping: 5-7 business days, free over $100
- 5-year warranty on furniture, 1-year on accessories
- VIP customers: priority support, free express shipping, 60-day returns
- Discount codes: WELCOME10, ERGOSAVE20, BUNDLE15, SORRY15
- Support agents have $200 refund limit; escalate above that

## How to Respond

Use EXACTLY this XML format for tool calls:
<tool_call>
<function=TOOL_NAME>
<parameter=PARAM_NAME>value</parameter>
</function>
</tool_call>

Strategy:
1. Look up the customer profile first
2. Check their order details
3. Send a helpful reply with a concrete solution
4. Resolve the ticket when the issue is addressed

Always investigate before replying. Be professional and empathetic."""


def format_initial_obs(obs):
    """Format the initial observation (from reset) as a user message."""
    return f"""New support ticket received.

{obs.ticket_context}

Customer message:
{obs.customer_message}

What tool would you like to use to help this customer?"""


def format_step_obs(obs):
    """Format a step observation (after tool call) as a user message."""
    parts = []
    if obs.tool_name:
        parts.append(f'Tool "{obs.tool_name}" result:')
        parts.append(obs.tool_result if obs.tool_result else "(no result)")
        parts.append("")
    if obs.customer_message:
        parts.append("Customer responded:")
        parts.append(obs.customer_message)
        parts.append("")
    parts.append(f"Satisfaction: {obs.satisfaction:.0%} | Steps: {obs.step_count}/10")
    parts.append("")
    parts.append("What would you like to do next?")
    return "\n".join(parts)

In [ ]:
def run_episode(env, generate_fn, seed=None):
    """Run one full episode, returning list of step records."""
    obs = env.reset(seed=seed) if seed else env.reset()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_initial_obs(obs)},
    ]

    steps = []
    ticket_id = obs.ticket_id

    while not obs.done:
        # Snapshot the prompt at this decision point
        prompt_snapshot = [dict(m) for m in messages]

        # Generate with the behavior policy
        response = generate_fn(messages)

        # Parse tool call from response
        tool_name, tool_args = parse_tool_call(response)
        if tool_name is None:
            # Fallback: treat raw text as a send_reply
            tool_name = "send_reply"
            tool_args = {"ticket_id": ticket_id, "message": response[:500]}

        # Ensure ticket_id is set for tools that need it
        if tool_name in ("send_reply", "update_ticket") and "ticket_id" not in tool_args:
            tool_args["ticket_id"] = ticket_id

        # Execute in environment
        obs = env.call_tool(tool_name, **tool_args)

        steps.append({
            "prompt": prompt_snapshot,
            "completion": response,
            "tool_name": tool_name,
            "tool_args": tool_args,
            "tool_result": obs.tool_result,
            "customer_message": obs.customer_message,
            "satisfaction": obs.satisfaction,
            "satisfaction_delta": obs.satisfaction_delta,
            "done": obs.done,
            "reward": obs.reward,
            "resolved": obs.resolved,
            "step_count": obs.step_count,
        })

        # Extend conversation for next turn
        messages.append({"role": "assistant", "content": response})
        if not obs.done:
            messages.append({"role": "user", "content": format_step_obs(obs)})

    # Backfill final episode reward to all steps
    final_reward = obs.reward
    for step in steps:
        step["episode_reward"] = final_reward
        step["episode_resolved"] = obs.resolved

    return steps


def format_grpo_dataset(all_steps):
    """Convert collected steps into GRPO training dataset."""
    records = []
    for step in all_steps:
        answer = json.dumps({
            "episode_reward": step["episode_reward"],
            "resolved": step["episode_resolved"],
            "valid_tools": list(VALID_TOOLS),
        })
        records.append({"prompt": step["prompt"], "answer": answer})
    return Dataset.from_list(records)

In [ ]:
# Collect episodes using OpenAI as behavior policy
# For production training, you'd use vLLM with Qwen 3.5 on a GPU instance

client = OpenAI()  # uses OPENAI_API_KEY from env

def generate_fn(messages):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7,
        max_tokens=512,
    )
    return resp.choices[0].message.content or ""


NUM_EPISODES = 8  # increase for real training (64+ recommended)

all_steps = []
env = CustomerSupportEnv(base_url=ENV_URL)

with env:
    for i in range(NUM_EPISODES):
        t0 = time.time()
        steps = run_episode(env, generate_fn, seed=42 + i)
        elapsed = time.time() - t0

        all_steps.extend(steps)
        final = steps[-1] if steps else {}
        reward = final.get("episode_reward", 0)
        resolved = final.get("episode_resolved", False)
        status = "RESOLVED" if resolved else "not resolved"
        print(f"Episode {i+1}/{NUM_EPISODES}: {len(steps)} steps, "
              f"reward={reward:.3f}, {status}, {elapsed:.1f}s")

print(f"\nCollected {len(all_steps)} total training steps")

# Format as GRPO dataset
dataset = format_grpo_dataset(all_steps)
print(f"GRPO dataset: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample prompt (first message):")
print(dataset[0]["prompt"][0]["content"][:200])

## 4. Reward Functions

GRPO generates multiple completions per prompt and uses reward functions to rank them. Our reward ensemble captures different aspects of good customer support behavior:

| Reward | Weight | What it measures |
|--------|--------|-----------------|
| `format_reward` | 1.0 | Valid XML tool call syntax |
| `tool_validity_reward` | 0.8 | Uses a real tool name |
| `reasoning_reward` | 0.5 | Thinks before acting |
| `no_reasoning_leak_reward` | 0.7 | Doesn't leak internal reasoning to customer |
| `action_quality_reward` | 0.5 | References relevant ground truth values |

In [ ]:
def _get_text(completion):
    """Extract text from completion (handles both list and string formats)."""
    if isinstance(completion, list):
        return completion[0]["content"] if completion else ""
    return str(completion)


def _parse_answer(ans):
    """Parse the answer JSON string."""
    if isinstance(ans, str):
        try:
            return json.loads(ans)
        except json.JSONDecodeError:
            return {}
    return ans if isinstance(ans, dict) else {}


# --- Reward Functions ---

def format_reward(completions, **kwargs) -> list[float]:
    """Valid XML tool call format?

    +1.0  valid <tool_call> with function + params
    +0.25 partial structure
    -1.0  empty or no tool call
    """
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        if not text.strip():
            rewards.append(-1.0)
            continue
        match = TOOL_CALL_RE.search(text)
        if match:
            params = PARAM_RE.findall(match.group(2))
            rewards.append(1.0 if params else 0.5)
        elif "<tool_call>" in text or "<function=" in text:
            rewards.append(0.25)
        else:
            rewards.append(-1.0)
    return rewards


def tool_validity_reward(completions, answer=None, **kwargs) -> list[float]:
    """Is the tool name valid?

    +1.0  valid tool name
    -0.5  parseable but wrong tool
     0.0  no tool call
    """
    rewards = []
    answers = answer or [None] * len(completions)
    for completion, ans in zip(completions, answers):
        text = _get_text(completion)
        ans_data = _parse_answer(ans) if ans else {}
        valid = set(ans_data.get("valid_tools", VALID_TOOLS))

        match = TOOL_CALL_RE.search(text)
        if match:
            tool_name = match.group(1).strip()
            rewards.append(1.0 if tool_name in valid else -0.5)
        else:
            rewards.append(0.0)
    return rewards


def reasoning_reward(completions, **kwargs) -> list[float]:
    """Does the completion have reasoning before the tool call?

    +1.0  reasoning text (>30 chars) before <tool_call>
    +0.3  has tool call but no reasoning
    -0.5  no tool call at all
    """
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        tool_pos = text.find("<tool_call>")
        if tool_pos > 30:
            rewards.append(1.0)
        elif tool_pos >= 0:
            rewards.append(0.3)
        else:
            rewards.append(-0.5)
    return rewards


def no_reasoning_leak_reward(completions, **kwargs) -> list[float]:
    """Penalize leaking reasoning into send_reply message.

    +1.0  send_reply message doesn't start with reasoning patterns
    -1.0  send_reply message starts with "The customer", "I need to", etc.
     0.0  not a send_reply (neutral)
    """
    reasoning_starts = [
        "The customer", "I need to", "Let me", "From the", "Based on",
        "Looking at", "According to", "I should", "First,", "Now I",
        "The user", "I'll", "I will",
    ]
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = TOOL_CALL_RE.search(text)
        if match and match.group(1).strip() == "send_reply":
            params = dict(PARAM_RE.findall(match.group(2)))
            msg = params.get("message", "").strip()
            if any(msg.startswith(p) for p in reasoning_starts):
                rewards.append(-1.0)
            else:
                rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def action_quality_reward(completions, answer=None, **kwargs) -> list[float]:
    """Heuristic quality: references ground truth values.

    +0.3 per ground truth referenced, capped at 1.0
    """
    rewards = []
    answers = answer or [None] * len(completions)
    for completion, ans in zip(completions, answers):
        text = _get_text(completion)
        ans_data = _parse_answer(ans) if ans else {}
        ground_truths = ans_data.get("ground_truth_values", [])
        score = 0.0
        for gt in ground_truths:
            if gt and str(gt).lower() in text.lower():
                score += 0.3
        rewards.append(min(1.0, score))
    return rewards


# Bundle them up
reward_funcs = [
    format_reward,
    tool_validity_reward,
    reasoning_reward,
    no_reasoning_leak_reward,
    action_quality_reward,
]
reward_weights = [1.0, 0.8, 0.5, 0.7, 0.5]

print(f"Reward functions: {[f.__name__ for f in reward_funcs]}")
print(f"Weights: {reward_weights}")

## 5. GRPO Training

Train with TRL's `GRPOTrainer`. For each prompt, the trainer:
1. Generates `num_generations` completions
2. Scores each with our reward functions
3. Uses the group-relative advantage to update the LoRA adapter

We use Qwen3-8B with LoRA (rank 32) — fits comfortably on a T4/A100.

In [ ]:
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

MODEL_NAME = "Qwen/Qwen3-8B"
OUTPUT_DIR = "./outputs/grpo-support-agent"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config — only train attention projections
peft_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# GRPO training config
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    num_generations=4,           # completions per prompt
    max_completion_length=512,
    temperature=0.7,
    beta=0.0,                    # no KL penalty
    loss_type="grpo",
    scale_rewards="group",
    bf16=True,
    logging_steps=5,
    save_steps=100,
    log_completions=True,
    max_grad_norm=1.0,
    warmup_ratio=0.1,
)

print(f"Model: {MODEL_NAME}")
print(f"LoRA rank: {peft_config.r}")
print(f"Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} grad accum")
print(f"Generations per prompt: {training_args.num_generations}")

In [ ]:
# Initialize trainer and run training
trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    args=training_args,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()

# Save the LoRA adapter
trainer.save_model(OUTPUT_DIR + "/final_adapter")
print(f"\nTraining complete! Adapter saved to {OUTPUT_DIR}/final_adapter")

## 6. Evaluate

Run the trained model on a few episodes and score with the same reward functions. Compare against the base model completions we collected earlier.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Load trained model for inference
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, OUTPUT_DIR + "/final_adapter")
model.eval()


def generate_trained(messages):
    """Generate with the trained model."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512, temperature=0.7,
            do_sample=True, top_p=0.9,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("Model loaded! Ready to evaluate.")

In [ ]:
def eval_episode_rewards(steps):
    """Score an episode's steps with all reward functions."""
    scores = {"format": [], "tool_valid": [], "reasoning": [], "no_leak": []}
    for s in steps:
        comp = s.get("completion", "")
        comp_list = [[{"role": "assistant", "content": comp}]]
        scores["format"].extend(format_reward(comp_list))
        scores["tool_valid"].extend(tool_validity_reward(comp_list))
        scores["reasoning"].extend(reasoning_reward(comp_list))
        scores["no_leak"].extend(no_reasoning_leak_reward(comp_list))
    return {k: sum(v) / len(v) if v else 0 for k, v in scores.items()}


# Run evaluation episodes with the trained model
NUM_EVAL = 4
eval_results = []

env = CustomerSupportEnv(base_url=ENV_URL)
with env:
    for i in range(NUM_EVAL):
        t0 = time.time()
        steps = run_episode(env, generate_trained, seed=77 + i)
        elapsed = time.time() - t0

        final = steps[-1] if steps else {}
        reward = final.get("episode_reward", 0)
        resolved = final.get("episode_resolved", False)
        scores = eval_episode_rewards(steps)

        eval_results.append({
            "episode": i + 1,
            "steps": len(steps),
            "reward": reward,
            "resolved": resolved,
            **scores,
        })

        status = "RESOLVED" if resolved else "not resolved"
        print(f"Episode {i+1}: {len(steps)} steps, reward={reward:.3f}, {status}")
        print(f"  Format: {scores['format']:.2f}, Valid: {scores['tool_valid']:.2f}, "
              f"Reasoning: {scores['reasoning']:.2f}, No-leak: {scores['no_leak']:.2f}")
        print()

# Summary
print("=" * 50)
print("EVALUATION SUMMARY (Trained Model)")
print("=" * 50)
avg_reward = sum(r["reward"] for r in eval_results) / len(eval_results)
resolve_rate = sum(1 for r in eval_results if r["resolved"]) / len(eval_results)
avg_format = sum(r["format"] for r in eval_results) / len(eval_results)
avg_valid = sum(r["tool_valid"] for r in eval_results) / len(eval_results)
avg_reasoning = sum(r["reasoning"] for r in eval_results) / len(eval_results)
avg_no_leak = sum(r["no_leak"] for r in eval_results) / len(eval_results)

print(f"Avg episode reward: {avg_reward:.3f}")
print(f"Resolution rate:    {resolve_rate:.0%}")
print(f"Avg format score:   {avg_format:.2f}")
print(f"Avg tool validity:  {avg_valid:.2f}")
print(f"Avg reasoning:      {avg_reasoning:.2f}")
print(f"Avg no-leak:        {avg_no_leak:.2f}")